<p style="text-align: right; margin: 0; line-height: 0;"><img width="100px" height="auto" src="../../.admin/img/eth_logo.png"></p>

<h1 style="text-align: center;">03 Mobile Systems</h1>
<hr>


## **9. 5G RedCap for the Future Internet of Things**

<div class="alert alert-light">

**Name:** (add your name here)

**Matriculation Number:** (add your number here)

</div>


### 9.1 Introduction

This notebook examines the application requirements that motivate RedCap and compares its capabilities with those of other cellular IoT technologies. It relates bandwidth, coding, spatial streams, and hardware complexity to achievable data throughput and examines the trade-offs involved in reducing device capabilities. The analysis provides a basis for selecting connectivity for wearables, industrial sensors, and other devices, while distinguishing Release 17 RedCap from Release 18 eRedCap. The discussion builds on [03.1 5G Mobile Communication](03.1_5G_Mobile_Communication.ipynb), [03.2 Cellular Networking](03.2_Cellular_Networking.ipynb), and the data-communication notebooks on modulation, noise, and Shannon capacity.

The Internet of Things (IoT) connects physical objects, such as meters, industrial sensors, and wearables, to computing systems that collect data or provide services. Sensors measure quantities such as temperature, movement, or energy consumption, while actuators can change a physical process, for example by opening a valve. Communication allows these objects to exchange measurements and commands with local gateways, application servers, or cloud services. An IoT system can use a local network without a direct public Internet connection; cellular connectivity is one option alongside technologies such as Wi-Fi and Bluetooth. The choice depends on data rate, coverage, latency, mobility, energy consumption, and cost.

Cellular IoT applications differ substantially in their communication requirements. A water meter may transmit a short report a few times per day, whereas a wearable exchanges notifications, audio, and software updates while moving between cells. A monitoring camera can generate sustained uplink traffic. These differences in traffic volume, mobility, latency, and energy constraints motivate device classes with different radio capabilities. Smartphone-class modems may provide capabilities that an IoT application does not require, increasing hardware complexity and potentially energy consumption.

<figure style="text-align: center; margin: 1.5em auto;">
  <img src="../../.admin/notebooks/picture/Connected%20Alpine%20Adventure.png" alt="A hiker uses a connected smartwatch in an Alpine landscape, with an illustrated wireless link to a cellular base station." style="width: 90%; height: auto; filter: grayscale(80%);">
  <figcaption><em>A wearable communicating with a cellular network.</em></figcaption>
</figure>

#### 9.1.1 Existing Cellular IoT Approaches

Narrowband Internet of Things (NB-IoT) and Long-Term Evolution for Machines (LTE-M) are cellular technologies derived from LTE and designed for low-power wide-area communication. NB-IoT uses a narrow radio channel to support small, infrequent data transfers, such as meter readings. LTE-M supports higher data rates and connected-mode handover, making it suitable for mobile trackers and devices with more frequent communication. Both provide power-saving mechanisms, but achievable battery life and coverage depend on the device, traffic pattern, and network configuration [1, 4]. LTE Cat-1 bis provides another LTE option for moderate data rates with a single receive antenna. Reduced Capability (RedCap), introduced in 3GPP Release 17, extends 5G New Radio (NR) to devices **with lower complexity requirements than conventional broadband terminals**. Release 18 introduces enhanced RedCap (eRedCap), which further reduces the supported peak data rate. These technologies address overlapping application requirements and coexist within the cellular ecosystem [1, 2, 3].

The table identifies technology families and selected capability scales. LTE-M here means the original Cat-M1 class; later capabilities and network configurations differ [1, 3, 4, 5].

| Technology | Bandwidth and capability context | Data-rate scale | Application considerations |
|---|---|---|---|
| NB-IoT | Narrowband radio, 180 kHz occupied bandwidth | Small reports, typically kbit/s scale; release and coverage matter | Metering and infrequent telemetry; evaluate access delay and mobility needs |
| LTE-M / Cat-M1 | 1.4 MHz device bandwidth | Around 1 Mbit/s class | Mobile trackers and sensors; power-saving behavior depends on configuration |
| LTE Cat-1 bis | LTE operation, up to 20 MHz; one receive antenna | 10 Mbit/s DL, 5 Mbit/s UL capability | Moderate-rate mobile IoT; check bands, service, and module integration |
| Release 17 RedCap | Up to 20 MHz in FR1 (0.410-7.125 GHz), 100 MHz in FR2 (24.25-71 GHz) | Tens to low hundreds of Mbit/s, configuration dependent | Wearables, industrial telemetry, and selected cameras or terminals |
| Release 18 eRedCap | FR1 (0.410-7.125 GHz); retains 20 MHz RF capability | Peak processing capability capped at 10 Mbit/s in each direction | Less demanding NR IoT workloads |
| Conventional 5G NR | Wider bandwidths, more layers and carrier combinations available | Hundreds of Mbit/s to Gbit/s under favorable configurations | Smartphones and demanding broadband workloads |

DL means downlink, from the network to the device; UL means uplink. RF denotes radio frequency. Frequency Range 1 (FR1) spans 410-7,125 MHz. Frequency Range 2 comprises FR2-1 (24.25-52.6 GHz) and FR2-2 (52.6-71 GHz), as defined in [3GPP TS 38.101-2, Table 5.1-1](https://www.etsi.org/deliver/etsi_ts/138100_138199/13810102/18.12.00_60/ts_13810102v181200p.pdf). These are NR frequency-range definitions, not a claim that a RedCap device supports every band or both FR2 subranges. The carrier frequency in GHz is distinct from the channel bandwidth in MHz. Compare a specific device and network, not just the family name. Low peak rate alone does not imply low energy per delivered bit.

#### 9.1.2 Commercial Example: Apple Watch and Network Support

Apple lists 5G RedCap support for cellular variants of Apple Watch Series 11, Apple Watch SE 3, and Apple Watch Ultra 3 [6, 7, 8]. These are examples of wearable devices balancing connectivity with size and battery constraints. This does not establish that every Apple Watch model, country, or subscription supports RedCap, nor does it establish eRedCap support.

A usable connection needs compatible bands, a RedCap-capable 5G standalone (SA) network, enabled network features, and provisioned service. Standalone means that NR connects to a 5G core without relying on LTE as the radio anchor. A dual-mode device may instead attach through LTE fallback; device capability therefore does not establish the access mode currently in use [2, 9].


### 9.2 Radio Technology

An FR1 RedCap implementation may use one or two receive branches. Two receive branches can improve reception through diversity without necessarily carrying two independent data streams. Two streams require both device support and a suitable spatial channel. Half-duplex frequency-division duplexing (FDD) is an available simplification: the device does not transmit and receive at the same time even though the uplink and downlink use separate frequency bands. Time-division duplexing (TDD) instead separates directions in time on one band. Duplex timing, scheduling, and device capabilities influence the usable UL/DL rates [2].

The modulation schemes and constellation diagrams used here are introduced in [02.4 Digital Modulation](../02_Data_Communication/02.4_Digital_Modulation.ipynb). For coding rate and error protection, see [02.3 Channel Coding](../02_Data_Communication/02.3_Channel_Coding.ipynb). Quadrature phase-shift keying (QPSK) and quadrature amplitude modulation (QAM) encode data in modulation symbols. For an $M$-point modulation constellation, each symbol carries $Q_m=\log_2 M$ coded bits: QPSK gives 2, 16-QAM gives 4, 64-QAM gives 6, and 256-QAM gives 8. The coding rate $r_c$ is the fraction of coded bits carrying information. Higher modulation and coding rates require suitable channel quality.

The model uses a pre-overhead spectral efficiency per stream $\eta$, measured in bit/s/Hz, to represent the information rate supported by the selected modulation and coding. Resource and protocol losses are represented separately by the overhead fraction $h$. We estimate payload data throughput as

$$R=B\eta N_s(1-h)q.$$

Within the resources allocated to the chosen direction, $h=0$ means that none of the pre-overhead information capacity is lost to the modeled overhead, so the fraction available for payload is $1-h=1$. At $h=1$, all of that capacity is consumed by overhead or unavailable resources, and payload throughput is zero. For example, $h=0.2$ leaves 80% available for payload. Coding redundancy is already accounted for in $\eta$, while $q$ describes the allocated resource share; neither should be counted again in $h$.

| Variable | Meaning | Unit or range |
|---|---|---|
| $B$ | Channel bandwidth | 5, 10, 15, or 20 MHz in the experiment; converted to Hz |
| $\eta$ | Information spectral efficiency per stream before the modeled overhead | 0.25-5.5 bit/s/Hz in the experiment |
| $N_s$ | Independent downlink spatial data streams | 1 or 2 |
| $h$ | Fraction lost to modeled overhead or unavailable resources within the allocated share | 0 to 1 |
| $q$ | Allocated downlink share of time/resources | 0 to 1 |
| $R$ | Estimated delivered payload rate | bit/s |

For $B=20$ MHz, $\eta=3$, $N_s=1$, $h=0.2$, and $q=1$, the estimate is 48 Mbit/s. The underlying 20 MHz, single-stream radio configuration is comparable in simplicity to a basic single-stream Wi-Fi 4 (IEEE 802.11n) implementation. Both use modest channel bandwidth and one spatial data stream, limiting the sampling bandwidth and spatial processing required compared with wider-band, multi-stream designs.
Release 18 eRedCap reduces the peak processing rate to 10 Mbit/s in each direction and is restricted to FR1. It retains 20 MHz RF bandwidth capability; an optional reduction to approximately 5 MHz equivalent data-channel baseband processing can further reduce complexity. The processing bandwidth and RF bandwidth describe different parts of the receiver. The experiment below represents Release 17 RedCap and does not model the eRedCap rate restriction [2, 3].


### 9.3 Interactive Throughput Experiment: FR1 RedCap Downlink

This experiment estimates Release 17 FR1 RedCap downlink throughput. The bandwidth control offers 5, 10, 15, and 20 MHz; the maximum is the FR1 RedCap limit. Supported bandwidth choices depend on the actual operating band and subcarrier spacing. The stream control offers one or two downlink spatial streams. Two streams require a device with suitable receive capabilities and a channel that supports two independent layers. RedCap does not support four downlink streams. Its uplink supports only one stream and is not represented by the two-stream setting ([RedCap capability overview](https://www.ericsson.com/en/reports-and-papers/white-papers/redcap-expanding-the-5g-device-ecosystem-for-consumers-and-industries)).

Run the launcher below in a Jupyter frontend with widget support. It uses NumPy, Matplotlib, and ipywidgets and requires no modem or network connection. The initial settings are 20 MHz, 1 bit/s/Hz, one stream, zero overhead, and full resource allocation, giving 20 Mbit/s. These differ from the worked 48 Mbit/s example in Section 9.2. Change one input at a time. The curve varies bandwidth within 5-20 MHz while holding the other controls fixed. The spectral efficiency range of 0.25-5.5 bit/s/Hz per stream is an illustrative teaching range, not a standardized list of modulation and coding schemes or a universal RedCap limit. The resource share $q$ applies to downlink; on a TDD carrier, uplink and downlink cannot each occupy all of the time simultaneously.

With the highest available bandwidth, spectral efficiency, and stream count, and with zero overhead and full downlink resource allocation, the estimate is

$$R_{\mathrm{max,model}}=20\times10^6\times5.5\times2\times(1-0)\times1
=220\times10^6\ \mathrm{bit/s}=220\ \mathrm{Mbit/s}.$$

The bandwidth and stream choices remain within FR1 RedCap hardware capabilities; the 220 Mbit/s result is the upper estimate for the selected teaching range, not a standardized RedCap peak rate. The result is still an idealized estimate: zero overhead, full resource allocation, and favorable radio conditions need not occur in practice. A particular modem's supported modulation, coding, and processing capabilities, together with network scheduling and retransmissions, determine its achievable throughput.


In [ ]:
from pathlib import Path

_helper = next(root / ".admin/notebooks/redcap_throughput.py"
               for root in (Path.cwd(), *Path.cwd().parents)
               if (root / ".admin/notebooks/redcap_throughput.py").is_file())
get_ipython().run_line_magic("run", f'-i "{_helper.as_posix()}"')


### 9.4 Use Cases

A factory can compare RedCap for mobile inspection terminals or condition-monitoring devices with LTE-M for small reports and conventional NR for demanding broadband. Private 5G may provide local coverage, subscriber control, and integration with an on-site core and application servers.

A possible architecture connects local sensors over Wi-Fi or mesh to a gateway, with cellular as the gateway's wide-area uplink or backup. Another equips each mobile device with cellular directly. The first can reduce subscriptions but adds a gateway dependency and shared bottleneck; the second adds per-device modem, antenna, and service costs. Mesh forwarding consumes airtime and energy. Failover can preserve connectivity while still interrupting sessions or changing latency. Illustrative Swiss or European applications include building telemetry, mobile maintenance terminals, logistics tracking, and temporary video monitoring.

In the course's [humanitarian communication scenario](../../01_Lectures/15%20Mobile%20Computing%20-%20Specialized%20Applications%20-%20Humanitarian%20Use.pdf), relief and rescue teams operate where communication infrastructure may be damaged, unavailable, or intermittent. A local Wi-Fi or mesh network can connect team members and sensors, while a gateway uses cellular connectivity when a working network is reachable. RedCap could serve a gateway with moderate traffic requirements, provided that compatible network access and service are available; a shared gateway carrying substantial traffic may require conventional NR instead. When the cellular path is unavailable, local communication can continue within the connected mesh, and delay-tolerant forwarding can store messages until an onward connection becomes available. This architecture requires sufficient local power, suitable relay placement, and protection of sensitive operational data. RedCap depends on cellular infrastructure and therefore cannot by itself restore connectivity where that infrastructure is absent.


### 9.5 References and Further Reading

1. [GSMA: Mobile IoT in the 5G Future - NB-IoT and LTE-M in the Context of 5G](https://www.gsma.com/solutions-and-impact/industries/smart-mobility/gsma_resources/mobile-iot-5g-future-2/). Background on the continuing role of cellular LPWA technologies.
2. [Ericsson: RedCap/eRedCap - standardizing simplified 5G IoT devices](https://www.ericsson.com/en/blog/2024/12/redcap-eredcap). Industry explanation of simplifications, duplexing, and eRedCap's RF/baseband distinction; not measured device performance.
3. [3GPP TS 38.306 v18.4.0, published by ETSI](https://www.etsi.org/deliver/etsi_ts/138300_138399/138306/18.04.00_60/ts_138306v180400p.pdf). Normative capability definitions: clauses 4.2.21 and 4.2.22. This is a pinned Release 18 version, not a claim to be the newest revision.
4. [u-blox: LTE-M](https://www.u-blox.com/en/technologies/lte-cat-m) and [NB-IoT](https://www.u-blox.com/en/technologies/narrowband-iot-nb-iot). Manufacturer overviews of bandwidth, use cases, and operating modes.
5. [u-blox: LTE Cat 1](https://www.u-blox.com/en/technologies/lte-cat-1). LTE category context and UL/DL capability scales.
6. [Apple Watch Series 11 technical specifications](https://support.apple.com/en-us/125093). 
7. [Apple Watch SE 3 technical specifications](https://support.apple.com/en-us/125094).
8. [Apple Watch Ultra 3 technical specifications](https://support.apple.com/en-us/125095).
9. [Quectel RG255C series](https://www.quectel.com/product/5g-redcap-rg255c-series/). Commercial RedCap module example; capabilities depend on the exact regional variant.
10. [Swiss OFCOM: frequency use and licenses for non-public networks](https://www.bakom.admin.ch/en/frequency-use-with-or-without-licences). Entry point for Swiss licensing conditions relevant to private 5G networks.
11. [Course reading: Keysight, Boost IoT with 5G NR RedCap (2023)](../../../BackgroundReading/5G%20Cellular/Keysight%20Study%20on%20RedCap/2023-boost-iot-with-5g-nr-redcap.pdf) and [Ericsson, RedCap: Expanding the 5G Device Ecosystem (2023)](../../../BackgroundReading/5G%20Cellular/Ericsson/2023-redcap-expanding-5g-device-ecosystem.pdf).
12. [ETH Library search guidance and swisscovery](https://library.ethz.ch/en/find-media/searching.html). Search for *Cellular IoT: From 5G to 5G Advanced* and for cellular IoT textbooks; check the catalog for editions and access options.


----
### 9.6 Hands-On Exercises

<div class="alert alert-block alert-info">Fill in your responses and save this notebook. See the <a href="../../03_Assignments/Submission-Guidelines.md">submission guidelines</a> for how to submit your work.</div>


**Question 01: Use the interactive graph to find the highest throughput. What is its value, and which control settings produce it?**

(answer here)


**Question 02: Why might a real RedCap device achieve a lower throughput than the maximum shown in the graph?**

(answer here)


**Question 03: OPTIONAL: Are all explanations and exercises in this notebook clear? Which parts, if any, need clarification or improvement?**

(answer here)
